# تصنيف طبي تعليمي — Breast Cancer Wisconsin

شغّلي Runtime → Run all. البيانات مضمّنة ولا يلزم رفع ملف أو اتصال إنترنت. **النموذج للتعلم فقط وليس للتشخيص الطبي.**


In [ ]:
# تصنيف تعليمي لبيانات Breast Cancer Wisconsin Diagnostic
# لا يحتاج إلى رفع بيانات أو اتصال إنترنت لتحميل مجموعة البيانات.
# تنبيه: النموذج تعليمي فقط وليس أداة تشخيص طبي.

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

# تحميل البيانات من مجموعة البيانات المضمنة في scikit-learn
# في المصدر الأصلي: target 0=malignant و1=benign.
data = load_breast_cancer(as_frame=True)
X = data.data
# إعادة الترميز حتى تكون الفئة الإيجابية 1 = خبيث
# هذا يسهل قراءة recall الخاص باكتشاف الحالات الخبيثة.
y = (data.target == 0).astype(int)
labels = ["حميد (Benign)", "خبيث (Malignant)"]

print("اسم مجموعة البيانات: Breast Cancer Wisconsin (Diagnostic)")
print("عدد الحالات:", X.shape[0])
print("عدد الخصائص:", X.shape[1])
print("توزيع الفئات (0=حميد، 1=خبيث):")
print(y.value_counts().sort_index().rename(index={0: labels[0], 1: labels[1]}))
print("\nأول 5 صفوف من الخصائص:")
print(X.head())

# تقسيم طبقي إلى تدريب واختبار، مع إبقاء الاختبار خارج اختيار النموذج
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# التحجيم والتصنيف داخل Pipeline يمنع تسرب بيانات الاختبار
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(probability=True, random_state=42))
])

# اختيار إعدادات SVM بواسطة تحقق متقاطع على بيانات التدريب فقط
params = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.01, 0.001],
    "classifier__kernel": ["rbf"]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline, params, scoring="accuracy", cv=cv, n_jobs=-1, refit=True
)
search.fit(X_train, y_train)

model = search.best_estimator_
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nأفضل الإعدادات:", search.best_params_)
print(f"متوسط الدقة في التحقق المتقاطع (على التدريب): {search.best_score_:.4f}")
print(f"دقة الاختبار المحجوز: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC للاختبار: {roc_auc_score(y_test, y_prob):.4f}")
print("\nتقرير التصنيف (الفئة الإيجابية = خبيث):")
print(classification_report(
    y_test, y_pred, target_names=labels, digits=4, zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=labels, cmap="Blues", values_format="d"
)
plt.title("مصفوفة الالتباس - بيانات الاختبار")
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()

print("\nتنبيه: هذا نموذج تعليمي على بيانات عامة، وليس تشخيصًا أو نصيحة طبية.")
